# HealthConnect ML Pipeline — Walkthrough Notebook

**AnalystLab Africa — HealthConnect Clinic Experience Lab**
**Track:** Machine Learning Engineering
**Weeks covered:** 5 (implementation) → 6 (integration) → 7 (testing & reliability)

This notebook is a **supporting artefact**, not the primary deliverable — the pipeline itself lives in `src/` as proper Python modules (so it's testable and importable), and the full write-up for each week is in `docs/`. This notebook exists to let a reviewer *see* the pipeline run end-to-end in one place, with real output, without having to open five separate files.

Run this notebook from the **repository root** (`healthconnect-ml-engineering/`), with the virtual environment from `requirements.txt` active.

Sections:
1. Load & validate the real dataset
2. Clean the data
3. Engineer features (target + engineered columns)
4. Train & compare candidate models (Week 6)
5. Score a batch with the recommended model (Week 6/7)
6. Week 7 reliability fix — before/after on a single-row batch
7. Run the automated test suite


## 0. Setup

Adds the repo root to the path (in case this notebook is opened from inside `notebooks/`) and imports the pipeline modules directly — the same modules used by `python -m src.training.train` etc.

In [1]:
import sys, os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # so relative paths (config.yaml, data/raw/...) resolve from repo root
sys.path.insert(0, ".")

import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

from src.data.validate import load_data, validate_schema, check_consistency, quality_report
from src.data.clean import clean_appointments
from src.features.build_features import build_target, build_feature_matrix, fit_feature_params
from src.training.train import MODEL_FACTORY, time_aware_split, evaluate, validate_training_inputs
from src.inference.score import score_batch, risk_tier
from src.config import get_config

cfg = get_config()
print("Working directory:", os.getcwd())
print("Config loaded. Raw data path:", cfg["data"]["raw_path"])


Working directory: /home/claude/healthconnect/repo/healthconnect-ml-engineering
Config loaded. Raw data path: data/raw/HealthConnect_Appointment_Data.csv


## 1. Load & Validate

Loads the real 5,000-row `HealthConnect_Appointment_Data.csv`, checks it against the Data Dictionary schema, and runs the cross-field consistency checks built in Week 5 (see `docs/PIPELINE.md` §2 for the full write-up of these findings — including a real discrepancy between the Data Dictionary and the actual date format).

In [2]:
raw = load_data(cfg["data"]["raw_path"])
print(f"Loaded {len(raw)} rows, {len(raw.columns)} columns.")
raw.head(3)


Loaded 5000 rows, 18 columns.


,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,waiting_time_minutes,appointment_outcome
0,HC-00001,P-1613,Female,39,35-44,Follow-up,2025-02-06,2025-02-18,Tuesday,Afternoon,12,2,0,Yes,WhatsApp,19.3,29.0,No-Show
1,HC-00002,P-0813,Male,31,25-34,Specialist Consultation,2026-02-25,2026-02-27,Friday,Morning,2,6,0,Yes,SMS,14.3,42.0,Attended
2,HC-00003,P-1366,Female,50,45-54,General Consultation,2025-11-16,2025-12-24,Wednesday,Morning,38,5,1,Yes,SMS,11.4,11.0,No-Show


In [3]:
issues = validate_schema(raw)
print("Schema issues:", issues if issues else "None — matches Data Dictionary column set.")

print("\nConsistency checks (0 = good):")
for check, count in check_consistency(raw).items():
    flag = "OK" if count == 0 else "REVIEW"
    print(f"  [{flag}] {check}: {count}")


Schema issues: None — matches Data Dictionary column set.

Consistency checks (0 = good):
  [OK] duplicate_rows: 0
  [OK] duplicate_appointment_id: 0
  [OK] booking_after_appointment: 0
  [OK] no_shows_exceed_previous_appointments: 0
  [OK] negative_booking_lead_days: 0
  [OK] negative_waiting_time: 0
  [OK] lead_days_mismatch_vs_dates: 0
  [OK] reminder_channel_set_when_not_sent: 0
  [REVIEW] waiting_time_present_for_non_attended: 2647


In [4]:
quality_report(raw)


,column,missing_count,missing_pct,allowed_missing
0,reminder_channel,1366,27.32,True
1,distance_to_clinic_km,90,1.80,True
2,waiting_time_minutes,60,1.20,True
3,appointment_id,0,0.00,False
4,patient_id,0,0.00,False
5,gender,0,0.00,False
6,booking_date,0,0.00,False
7,age,0,0.00,False
8,age_group,0,0.00,False
9,appointment_type,0,0.00,False


## 2. Clean

Imputes `distance_to_clinic_km` (median by `age_group`), fills `reminder_channel` with an explicit `"None"` category (verified structural, not random missingness), and leaves `waiting_time_minutes` untouched here — it's excluded later at the feature stage, not the cleaning stage, since it's a *modelling* exclusion (data-leakage risk) rather than a *data-quality* one.

In [5]:
cleaned = clean_appointments(raw)
print(f"distance_to_clinic_km missing: {raw['distance_to_clinic_km'].isna().sum()} -> {cleaned['distance_to_clinic_km'].isna().sum()}")
print(f"reminder_channel missing: {raw['reminder_channel'].isna().sum()} -> {cleaned['reminder_channel'].isna().sum()}")
cleaned[["distance_to_clinic_km", "reminder_channel", "distance_was_missing"]].head(3)


distance_to_clinic_km missing: 90 -> 0
reminder_channel missing: 1366 -> 0


,distance_to_clinic_km,reminder_channel,distance_was_missing
0,19.3,WhatsApp,0
1,14.3,SMS,0
2,11.4,SMS,0


## 3. Feature Engineering

Builds the binary target (`is_no_show`), drops `Cancelled` appointments entirely (a cancellation is a different event from a no-show — see `docs/PIPELINE.md` §4.1), and constructs the feature matrix.

**Week 7 note:** `feature_params` (fixed category levels + a fixed `long_lead_time` threshold) must be fit once on a reference/training set and reused — passing `None` here would reproduce the two batch-size bugs documented in `docs/ISSUE_LOG.md` #11/#12. This notebook fits it on the full cleaned dataset since we're inspecting the whole pipeline, not simulating a specific train/test split yet (that happens in the next section).

In [6]:
targeted = build_target(cleaned)
print(f"Rows after dropping Cancelled: {len(targeted)} (removed {len(cleaned) - len(targeted)})")
targeted["is_no_show"].value_counts(normalize=True).round(3)


Rows after dropping Cancelled: 4737 (removed 263)


is_no_show
1    0.512
0    0.488
Name: proportion, dtype: float64

In [7]:
feature_params = fit_feature_params(targeted)
features = build_feature_matrix(targeted, feature_params)
print("Feature matrix shape:", features.shape)
features.head(3)


Feature matrix shape: (4737, 31)


,age,booking_lead_days,distance_to_clinic_km,distance_was_missing,is_no_show,no_show_rate_history,is_first_time_patient,is_weekend_appointment,long_lead_time,gender_Male,gender_Prefer not to say,age_group_25-34,age_group_35-44,age_group_45-54,age_group_55-64,age_group_65+,appointment_type_Follow-up,appointment_type_General Consultation,appointment_type_Specialist Consultation,appointment_day_Monday,appointment_day_Saturday,appointment_day_Sunday,appointment_day_Thursday,appointment_day_Tuesday,appointment_day_Wednesday,appointment_time_Evening,appointment_time_Morning,reminder_sent_Yes,reminder_channel_None,reminder_channel_SMS,reminder_channel_WhatsApp
0,39,12,19.3,0,1,0.0,0,0,0,False,False,False,True,False,False,False,True,False,False,False,False,False,False,True,False,False,False,True,False,False,True
1,31,2,14.3,0,0,0.0,0,0,0,True,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,True,False,True,False
2,50,38,11.4,0,1,0.2,0,0,0,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,True,False,True,True,False,True,False


## 4. Train & Compare Candidate Models (Week 6)

Three models are trained via the pluggable `MODEL_FACTORY` on an identical **time-aware split** (chronological on `appointment_date`, not random — see `docs/PIPELINE.md` §4). This mirrors exactly what `python -m src.training.train` does.

In [8]:
train_raw, test_raw, cutoff = time_aware_split(targeted, cfg["split"]["date_column"], cfg["split"]["test_fraction"])
print(f"Split cutoff: {cutoff.date()}  |  train={len(train_raw)}  test={len(test_raw)}")

train_params = fit_feature_params(train_raw)  # fit on TRAIN ONLY — avoids leaking test distribution
train_feat = build_feature_matrix(train_raw, train_params)
test_feat = build_feature_matrix(test_raw, train_params)

y_train = train_raw["is_no_show"].values
y_test = test_raw["is_no_show"].values
X_train = train_feat.drop(columns=["is_no_show"])
X_test = test_feat.drop(columns=["is_no_show"])
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

validate_training_inputs(X_train, X_test, y_train, y_test)
print("Input validation passed.")


17:53:06 | INFO | training | Input validation passed: 3789 train rows, 948 test rows, 30 features


Split cutoff: 2026-03-13  |  train=3789  test=948
Input validation passed.


In [9]:
results = {}
for name, factory in MODEL_FACTORY.items():
    model = factory()
    model.fit(X_train, y_train)
    metrics, cm = evaluate(model, X_test, y_test)
    results[name] = metrics
    print(f"{name:20s} -> {metrics}")

results_df = pd.DataFrame(results).T
results_df


baseline_logreg      -> {'accuracy': 0.6203, 'precision': 0.6268, 'recall': 0.6294, 'f1': 0.6281, 'roc_auc': 0.6733}
refined_logreg       -> {'accuracy': 0.6234, 'precision': 0.6376, 'recall': 0.6046, 'f1': 0.6206, 'roc_auc': 0.6734}


random_forest        -> {'accuracy': 0.6224, 'precision': 0.6299, 'recall': 0.6273, 'f1': 0.6286, 'roc_auc': 0.6724}


,accuracy,precision,recall,f1,roc_auc
baseline_logreg,0.6203,0.6268,0.6294,0.6281,0.6733
refined_logreg,0.6234,0.6376,0.6046,0.6206,0.6734
random_forest,0.6224,0.6299,0.6273,0.6286,0.6724


**Honest interpretation** (see `docs/PIPELINE.md` §9.3 for the full discussion): the "improved" model beats the baseline by ~0.0001 ROC-AUC — not a meaningful difference. The real value of the refinement was removing a multicollinearity issue (two features with contradictory coefficient signs), which improved interpretability at equal performance, not a modelling breakthrough. This is reported as-is rather than rounded up into a bigger claim than the data supports.

## 5. Score a Batch (Week 6/7)

Uses the actual trained model + fitted `feature_params` to score a batch of appointments, exactly as `python -m src.inference.score` does — including the Week 6 output validation (`validate_scores`, called inside `score_batch`) and the Week 7 fix for batch-size-consistent feature encoding.

In [10]:
recommended_name = "refined_logreg"  # matches models/model_registry_log.csv "candidate_recommended" entry
model = MODEL_FACTORY[recommended_name]()
model.fit(X_train, y_train)
feature_columns = list(model.feature_names_in_)

batch = raw.sort_values("appointment_date").tail(20)
scored = score_batch(batch, model, feature_columns, cfg, train_params)
scored


17:53:07 | INFO | inference | Output validation passed for 19 scored rows.


,appointment_id,no_show_probability,risk_tier
0,HC-01117,0.6549,Medium
1,HC-04622,0.5205,Medium
2,HC-02520,0.4463,Medium
3,HC-00401,0.6314,Medium
4,HC-01346,0.6906,High
5,HC-04503,0.2826,Low
6,HC-02173,0.7119,High
7,HC-00691,0.4536,Medium
8,HC-00810,0.4158,Medium
9,HC-03649,0.5887,Medium


In [11]:
scored["risk_tier"].value_counts()


risk_tier
Medium    12
High       4
Low        3
Name: count, dtype: int64

## 6. Week 7 Reliability Fix — Before/After on a Single-Row Batch

This is the concrete evidence for the two bugs documented in `docs/ISSUE_LOG.md` #11 and #12. **"Before"** simulates the old behaviour (no fitted `feature_params` — encoding derived from whatever's in the batch). **"After"** uses the fix (fitted params passed in).

**Important methodological note:** the single row below is cleaned *fresh*, exactly as `score.py` does with a real incoming batch — not sliced from an already-cleaned full dataset. Slicing after the fact would hide the bug entirely, because pandas' `Categorical` dtype remembers the full category list inherited from the larger frame it was created on. That's a realistic trap worth knowing about, not just a notebook technicality — it's the same trap that made this bug easy to miss during Week 5/6 testing, since testing was always done on the full dataset.

In [12]:
# Reproduce EXACTLY what happens in production: score.py calls clean_appointments()
# on the raw batch itself, not on a slice of an already-cleaned full dataset.
# (Slicing a single row AFTER cleaning the full dataset would hide this bug, because
# pandas' Categorical dtype remembers the full category list from the larger frame —
# that's a trap, not a fix, so we deliberately avoid it here.)

raw_single_row = raw.iloc[[0]]  # a single row, straight from the raw, uncleaned data
cleaned_single = clean_appointments(raw_single_row)
targeted_single = build_target(cleaned_single)

# BEFORE (bug reproduction): no feature_params passed in
before = build_feature_matrix(targeted_single)  # feature_params=None -> old, buggy behaviour
print("BEFORE fix — single-row feature matrix shape:", before.shape)

# AFTER (fix): fitted feature_params passed in
after = build_feature_matrix(targeted_single, feature_params)
print("AFTER fix  — single-row feature matrix shape:", after.shape)

print(f"\nFull-dataset feature matrix has {features.shape[1]} columns.")
print(f"BEFORE fix: single-row batch has {before.shape[1]} columns ({'MISMATCH — bug reproduced' if before.shape[1] != features.shape[1] else 'match'})")
print(f"AFTER fix:  single-row batch has {after.shape[1]} columns ({'MISMATCH' if after.shape[1] != features.shape[1] else 'match — fix confirmed'})")


BEFORE fix — single-row feature matrix shape: (1, 9)


AFTER fix  — single-row feature matrix shape: (1, 31)

Full-dataset feature matrix has 31 columns.
BEFORE fix: single-row batch has 9 columns (MISMATCH — bug reproduced)
AFTER fix:  single-row batch has 31 columns (match — fix confirmed)


In [13]:
# Issue #12 demonstration: long_lead_time for a short-lead-time appointment, single-row batch
raw_short_lead = raw.iloc[[0]].copy()
raw_short_lead["booking_lead_days"] = 1  # a genuinely short lead time
cleaned_short = clean_appointments(raw_short_lead)
targeted_short = build_target(cleaned_short)

before_llt = build_feature_matrix(targeted_short)["long_lead_time"].iloc[0]
after_llt = build_feature_matrix(targeted_short, feature_params)["long_lead_time"].iloc[0]

print(f"BEFORE fix: long_lead_time = {before_llt}  (bug: always 1 for a single-row batch, regardless of actual value)")
print(f"AFTER fix:  long_lead_time = {after_llt}  (correct: 0, since 1 day is a short lead time)")


BEFORE fix: long_lead_time = 1  (bug: always 1 for a single-row batch, regardless of actual value)
AFTER fix:  long_lead_time = 0  (correct: 0, since 1 day is a short lead time)


Both cells above reproduce the exact bugs found during Week 7 adversarial testing, and confirm the fix resolves them. Full root-cause analysis: `docs/PIPELINE.md` §10.

## 7. Run the Automated Test Suite

The notebook cells above are for *inspection* — the actual regression protection lives in `tests/`, run here for completeness. (Equivalent to running `pytest tests/ -v` from a terminal.)

In [14]:
import subprocess
result = subprocess.run(
    ["python3", "-m", "pytest", "tests/", "-v", "--no-header"],
    capture_output=True, text=True,
)
print(result.stdout[-3000:])  # last part of output — summary line included


st_validate_scores_rejects_out_of_range_probability PASSED [ 27%]
tests/test_integration.py::test_validate_scores_rejects_duplicate_appointment_id PASSED [ 29%]
tests/test_integration.py::test_risk_tier_thresholds_match_config PASSED [ 32%]
tests/test_integration.py::test_latest_recommended_model_path_resolves PASSED [ 35%]
tests/test_integration.py::test_score_batch_end_to_end PASSED            [ 37%]
tests/test_integration.py::test_single_row_batch_produces_full_column_set PASSED [ 40%]
tests/test_integration.py::test_skewed_batch_all_same_category_produces_full_column_set PASSED [ 43%]
tests/test_integration.py::test_long_lead_time_threshold_is_fixed_not_recomputed_per_batch PASSED [ 45%]
tests/test_integration.py::test_unseen_category_does_not_crash_and_is_logged PASSED [ 48%]
tests/test_integration.py::test_empty_batch_scores_without_error PASSED  [ 51%]
tests/test_integration.py::test_all_cancelled_batch_scores_without_error PASSED [ 54%]
tests/test_integration.py::test_missing_r

## Summary

| Stage | Status |
|---|---|
| Data validation | 5,000 rows, schema matches, 1 documentation discrepancy found (date format) |
| Cleaning | 0 missing values remaining in imputed columns |
| Feature engineering | 30-column feature matrix, leakage-prone columns excluded and verified absent |
| Model comparison | 3 candidates compared; `refined_logreg` recommended (marginal, honestly-reported improvement) |
| Batch inference | Validated output (probability range, no duplicates, valid risk tiers) |
| Week 7 reliability fix | 2 real train/serve skew bugs found and fixed — demonstrated above |
| Test suite | 37 automated tests |

Full detail for every claim above is in `docs/PIPELINE.md`, `docs/ISSUE_LOG.md`, and the weekly `docs/*.docx` reports.
